# 06 — Do arquivo ao céu variável: alertas em tempo real

Workshop SAB — AI-Scope / ADSS

**Pergunta:** como organizar uma decisão científica quando os eventos chegam em fluxo? Aqui usamos um replay finito para que a demonstração termine sozinha. As categorias e os campos do payload são descobertos na instância atual.

In [1]:
import json
import os
from datetime import datetime, timezone

import adss
import pandas as pd

cl = adss.ADSSClient(base_url="https://ai-scope.cbpf.br/")

## Categorias disponíveis

Categorias podem mudar entre instâncias. O replay abaixo usa `all` e limita a 20 alertas; `follow=False` encerra após os eventos retidos. Em produção, um consumidor contínuo precisa de política de reconexão, deduplicação e checkpoint.

In [21]:
categories = pd.DataFrame(cl.get_alert_categories())
display(categories)

alerts = list(cl.listen_alerts(categories="rapid-review", replay="earliest", limit=20, follow=False))
print(f"Alertas recebidos: {len(alerts)}")
if alerts:
    print("Campos do primeiro alerta:", sorted(alerts[0]))
    print(json.dumps(alerts[0], ensure_ascii=False, indent=2, default=str)[:3000])
else:
    print("Sem alertas retidos no momento; os exercícios abaixo continuam conceituais.")

,name,description,created_at
0,all,All Roman Alerts,2026-03-30T13:28:09.407189
1,rapid-candidates,Managed by Arandu routing policy,2026-10-26T14:12:39.324142
2,rapid-high-confidence,High-confidence transient candidates from RAPID,2026-09-15T19:40:06.525770
3,rapid-lightcurve-updates,Managed by Arandu routing policy,2026-10-26T14:12:39.414482
4,rapid-rejected,Managed by Arandu routing policy,2026-10-26T14:12:39.227482
5,rapid-review,Managed by Arandu routing policy,2026-10-26T14:12:39.198005
6,rapid-slow-review,Managed by Arandu routing policy,2026-10-26T14:12:39.389415
7,rapid-triage,Managed by Arandu routing policy,2026-10-26T14:12:39.130735


Alertas recebidos: 20
Campos do primeiro alerta: ['alert_id', 'category', 'dec', 'filter_band', 'magnitude', 'mjd', 'payload', 'ra', 'source']
{
  "alert_id": "9532840042",
  "source": "RAPID",
  "category": "rapid-review",
  "ra": 43.77505107798993,
  "dec": -2.6008087694650612,
  "mjd": 61339.59372548147,
  "magnitude": 26.98094213223191,
  "filter_band": "F129",
  "payload": {
    "snr": 6.774188041687012,
    "psfFlux": 58.5665168762207,
    "psfFluxErr": 8.645540237426758,
    "extendedness": 0.3317882716655731,
    "sharpness": 0.3478177487850189,
    "roundness1": 0.3992100656032562,
    "roundness2": -0.9729577898979187,
    "flags": 0,
    "sca": 1,
    "field": 3441402,
    "hp6": 44355,
    "hp9": 2304340,
    "cutouts": {
      "science": "https://arandu-portal.cbpf.br/api/cutouts/9532840042/science.fits",
      "template": "https://arandu-portal.cbpf.br/api/cutouts/9532840042/template.fits",
      "difference": "https://arandu-portal.cbpf.br/api/cutouts/9532840042/differen

## Inventário antes de priorizar

Não presumimos que todo evento tem magnitude, classe ou coordenadas. Primeiro medimos a presença dos campos. Só depois vale escrever uma regra dependente do esquema real. Um ID de evento e o horário de recebimento permitem rastrear decisões e evitar processar o mesmo replay duas vezes.

In [3]:
if alerts:
    inventory = pd.DataFrame([
        {"field": field, "present": sum(isinstance(a, dict) and a.get(field) is not None for a in alerts)}
        for field in sorted(set().union(*(a.keys() for a in alerts if isinstance(a, dict))))
    ]).sort_values(["present", "field"], ascending=[False, True])
    display(inventory)
else:
    inventory = pd.DataFrame(columns=["field", "present"])

received_utc = datetime.now(timezone.utc).isoformat()
print("Replay analisado em", received_utc)

,field,present
0,alert_id,20
1,category,20
2,dec,20
3,filter_band,20
4,magnitude,20
5,mjd,20
6,payload,20
7,ra,20
8,source,20


Replay analisado em 2026-10-21T13:26:19.729743+00:00


## Exercício de triagem

1. Escolha do inventário um identificador estável, posição, instante e sinais de confiança. Que campos estão ausentes?
2. Defina uma regra de prioridade com critérios explícitos e uma saída `review`, `follow_up` ou `ignore`. Qual critério exige consulta adicional ao AI-Scope?
3. Enriqueça poucos candidatos com `cone()` em Gaia ou com cutouts, como nos notebooks 04 e 05. Registre raio, tabela, versão da regra/modelo e horário da decisão.
4. Repita o replay: sua regra produz a mesma decisão por ID? Descreva como evitar duplicatas e como reavaliar um evento quando chega nova informação.

O replay não reproduz latência nem taxa real de chegada; meça esses fatores antes de implementar um consumidor contínuo.